# C02 — Shared incorrect result analysis

Investigate why GPT-OSS 20B and Dots returned 0 cents for C02,
while the reference value is 590320 cents.

C02 asks for the total revenue from all orders placed by customers
who are exactly 25 years old on 25 September 2026.

This analysis uses stored TD1 traces and makes no new model calls.

In [1]:
import csv
from pathlib import Path

# Locate the repository from a notebook started in the repository, TD2, or TD2/pair-b.
root = Path.cwd()
while not (root / "TD1" / "kit-td1" / "traces-td1.csv").is_file():
    if root == root.parent:
        raise FileNotFoundError("Run from the repository, TD2, or TD2/pair-b")
    root = root.parent

trace_path = root / "TD1" / "kit-td1" / "traces-td1.csv"

with trace_path.open(encoding="utf-8-sig", newline="") as stream:
    rows = list(csv.DictReader(stream))

# Display every stored attempt before selecting the final real attempts.
for row in rows:
    if row["case_id"].strip() == "C02":
        for field in (
            "provider", "model_requested", "date_utc", "attempts",
            "nature", "value", "reference", "verdict", "sql"
        ):
            print(f"{field}: {row.get(field, '')}")
        print("-" * 70)

provider: groq
model_requested: openai/gpt-oss-120b
date_utc: 2026-10-21 13:51:16
attempts: 1
nature: real
value: 590320
reference: 590320
verdict: correct
sql: SELECT COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN product p ON o.product_id = p.product_id
WHERE (strftime('%Y', '2026-09-25') - strftime('%Y', c.birth_date))
      - (strftime('%m-%d', '2026-09-25') < strftime('%m-%d', c.birth_date)) = 25;
----------------------------------------------------------------------
provider: groq
model_requested: openai/gpt-oss-20b
date_utc: 2026-10-21 13:51:21
attempts: 1
nature: real
value: 0
reference: 590320
verdict: incorrect
sql: SELECT COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN product p ON o.product_id = p.product_id
WHERE c.birth_date > '2025-09-25' AND c.birth_date <= '2026-09-25';
-------------------------------

In [2]:
import sqlite3

database_path = root / "TD1" / "kit-td1" / "data" / "boutique.sqlite"

# Open the database in read-only mode.
with sqlite3.connect(
    database_path.resolve().as_uri() + "?mode=ro",
    uri=True,
) as connection:

    # Re-run the stored real SQL queries for C02.
    for row in rows:
        if (
            row["case_id"].strip() == "C02"
            and row.get("nature", "").strip() == "real"
        ):
            result = connection.execute(row["sql"]).fetchone()[0]

            print(f"Model: {row['model_requested']}")
            print(f"Stored result: {row['value']}")
            print(f"Reproduced result: {result}")
            print("-" * 50)

Model: openai/gpt-oss-120b
Stored result: 590320
Reproduced result: 590320
--------------------------------------------------
Model: openai/gpt-oss-20b
Stored result: 0
Reproduced result: 0
--------------------------------------------------
Model: cohere/north-mini-code:free
Stored result: 321340
Reproduced result: 321340
--------------------------------------------------
Model: qwen/qwen3.8-27b
Stored result: 590320
Reproduced result: 590320
--------------------------------------------------
Model: dots-studio/dots-3-note-preview:free
Stored result: 0
Reproduced result: 0
--------------------------------------------------


In [3]:
# List eligible customers and their revenue across all orders.
diagnostic_sql = """
SELECT
    c.customer_id,
    c.birth_date,
    SUM(o.quantity_product * p.price) AS revenue_cents
FROM customers AS c
JOIN orders AS o ON o.customer_id = c.customer_id
JOIN product AS p ON p.product_id = o.product_id
WHERE c.birth_date > '2000-09-25'
  AND c.birth_date <= '2001-09-25'
GROUP BY c.customer_id, c.birth_date
ORDER BY c.birth_date, c.customer_id;
"""

with sqlite3.connect(
    database_path.resolve().as_uri() + "?mode=ro",
    uri=True,
) as connection:
    customer_rows = connection.execute(diagnostic_sql).fetchall()

print("customer_id | birth_date | revenue_cents")

for customer_id, birth_date, revenue_cents in customer_rows:
    print(f"{customer_id} | {birth_date} | {revenue_cents}")

total = sum(row[2] for row in customer_rows)
print(f"\nTotal revenue_cents: {total}")

customer_id | birth_date | revenue_cents
C005 | 2000-09-26 | 150240
C007 | 2000-11-30 | 118740
C009 | 2001-03-10 | 61330
C008 | 2001-05-15 | 99090
C002 | 2001-09-24 | 25920
C001 | 2001-09-25 | 135000

Total revenue_cents: 590320


In [4]:
# Identify valid C02 order lines excluded by the Dots filters.
dots_delta_sql = """
SELECT
    c.customer_id,
    c.birth_date,
    o.date AS order_date,
    SUM(o.quantity_product * p.price) AS omitted_revenue_cents
FROM customers AS c
JOIN orders AS o ON o.customer_id = c.customer_id
JOIN product AS p ON p.product_id = o.product_id
WHERE c.birth_date > '2000-09-25'
  AND c.birth_date <= '2001-09-25'
  AND NOT (
      c.birth_date = '2001-09-25'
      AND o.date >= '2026-09-25 00:00:00'
      AND o.date < '2026-09-26 00:00:00'
  )
GROUP BY c.customer_id, c.birth_date, o.date
ORDER BY c.customer_id, o.date;
"""

with sqlite3.connect(
    database_path.resolve().as_uri() + "?mode=ro",
    uri=True,
) as connection:
    dots_delta = connection.execute(dots_delta_sql).fetchall()

print("customer_id | birth_date | order_date | omitted_revenue_cents")

for record in dots_delta:
    print(" | ".join(str(value) for value in record))

print(
    "\nTotal omitted revenue_cents:",
    sum(record[3] for record in dots_delta),
)

customer_id | birth_date | order_date | omitted_revenue_cents
C001 | 2001-09-25 | 2026-01-15 19:30:43 | 25880
C001 | 2001-09-25 | 2026-01-28 03:54:01 | 23730
C001 | 2001-09-25 | 2026-02-12 14:42:43 | 16720
C001 | 2001-09-25 | 2026-03-01 00:00:00 | 17460
C001 | 2001-09-25 | 2026-03-03 17:58:02 | 6970
C001 | 2001-09-25 | 2026-03-04 11:45:15 | 12450
C001 | 2001-09-25 | 2026-03-06 00:19:36 | 4470
C001 | 2001-09-25 | 2026-04-10 08:28:49 | 27320
C002 | 2001-09-24 | 2026-05-29 03:52:20 | 4470
C002 | 2001-09-24 | 2026-05-31 23:59:59 | 13970
C002 | 2001-09-24 | 2026-09-07 12:51:43 | 7480
C005 | 2000-09-26 | 2026-01-06 21:23:30 | 22210
C005 | 2000-09-26 | 2026-01-17 08:58:53 | 16200
C005 | 2000-09-26 | 2026-03-19 23:54:09 | 6250
C005 | 2000-09-26 | 2026-04-08 21:53:59 | 11290
C005 | 2000-09-26 | 2026-06-01 10:30:00 | 29510
C005 | 2000-09-26 | 2026-07-03 18:41:34 | 12250
C005 | 2000-09-26 | 2026-07-04 12:46:44 | 9950
C005 | 2000-09-26 | 2026-08-02 00:33:03 | 27230
C005 | 2000-09-26 | 2026-08-24 1

In [5]:
# Save the diagnostic query and its executed results.
output_folder = root / "TD2" / "pair-b"

(output_folder / "c02-dots-delta.sql").write_text(
    dots_delta_sql,
    encoding="utf-8",
)

with (output_folder / "c02-dots-delta.csv").open(
    "w", encoding="utf-8", newline=""
) as stream:
    writer = csv.writer(stream)
    writer.writerow([
        "customer_id",
        "birth_date",
        "order_date",
        "omitted_revenue_cents",
    ])
    writer.writerows(dots_delta)

print("Saved c02-dots-delta.sql and c02-dots-delta.csv")

Saved c02-dots-delta.sql and c02-dots-delta.csv


The stored C02 results were successfully reproduced on the database.
The correct reference is 590320 cents, generated by six eligible customers.

The Dots query returns 0 because it applies two incorrect restrictions:
it selects only customers born on 25 September 2001 and only orders placed
on 25 September 2026. C02 instead requires all orders from customers aged
exactly 25 on the reference date.

The diagnostic query identifies 590320 cents of excluded revenue:
455320 cents from the five customers excluded by the birth-date restriction,
plus 135000 cents from C001's orders excluded by the order-date restriction.
This explains the entire gap with the reference. The SQL runs successfully,
but its filters answer a different question.

In [6]:
# Identify valid C02 sales excluded by the GPT-OSS 20B birth-date filter.
oss20b_delta_sql = """
SELECT
    c.customer_id,
    c.birth_date,
    SUM(o.quantity_product * p.price) AS omitted_revenue_cents
FROM customers AS c
JOIN orders AS o ON o.customer_id = c.customer_id
JOIN product AS p ON p.product_id = o.product_id
WHERE c.birth_date > '2000-09-25'
  AND c.birth_date <= '2001-09-25'
  AND NOT (
      c.birth_date > '2025-09-25'
      AND c.birth_date <= '2026-09-25'
  )
GROUP BY c.customer_id, c.birth_date
ORDER BY c.customer_id;
"""

with sqlite3.connect(
    database_path.resolve().as_uri() + "?mode=ro",
    uri=True,
) as connection:
    oss20b_delta = connection.execute(oss20b_delta_sql).fetchall()

print("customer_id | birth_date | omitted_revenue_cents")

for record in oss20b_delta:
    print(" | ".join(str(value) for value in record))

print(
    "\nTotal omitted revenue_cents:",
    sum(record[2] for record in oss20b_delta),
)

customer_id | birth_date | omitted_revenue_cents
C001 | 2001-09-25 | 135000
C002 | 2001-09-24 | 25920
C005 | 2000-09-26 | 150240
C007 | 2000-11-30 | 118740
C008 | 2001-05-15 | 99090
C009 | 2001-03-10 | 61330

Total omitted revenue_cents: 590320


In [7]:
# Save the GPT-OSS 20B diagnostic query and results.
output_folder = root / "TD2" / "pair-b"

(output_folder / "c02-20b-delta.sql").write_text(
    oss20b_delta_sql, encoding="utf-8"
)

with (output_folder / "c02-20b-delta.csv").open(
    "w", encoding="utf-8", newline=""
) as stream:
    writer = csv.writer(stream)
    writer.writerow([
        "customer_id", "birth_date", "omitted_revenue_cents"
    ])
    writer.writerows(oss20b_delta)

print("Saved c02-20b-delta.sql and c02-20b-delta.csv")

Saved c02-20b-delta.sql and c02-20b-delta.csv


The GPT-OSS 20B query uses the birth-date interval
(2025-09-25, 2026-09-25], selecting customers under one year old
on the reference date instead of customers aged 25.

The diagnostic query shows that all six eligible customers are excluded,
representing 590320 cents of omitted revenue. This accounts for the entire
gap between the reproduced model result of 0 and the reference of 590320.

GPT-OSS 20B and Dots therefore share the same incorrect result, but reach
it through different SQL errors. Agreement on a value is not evidence
that the value or the underlying reasoning is correct.

## C02 — Quorum decisions

The five models returned:

| Model | Value (cents) | Verdict |
|---|---:|---|
| GPT-OSS 120B | 590320 | Correct |
| GPT-OSS 20B | 0 | Incorrect |
| Cohere North Mini Code | 321340 | Incorrect |
| Qwen | 590320 | Correct |
| Dots | 0 | Incorrect |

### Fixed 2-of-3 panel

The fixed panel contains GPT-OSS 120B, Cohere North Mini Code and Qwen.

GPT-OSS 120B and Qwen agree on 590320 cents. This reaches the threshold
of two votes, so the panel accepts 590320. Comparison with the independent
reference confirms that the decision is correct.

### 3-of-5 quorum

Across all five models, 590320 receives two votes, 0 receives two votes,
and 321340 receives one vote.

No value reaches the threshold of three votes, so the quorum abstains.
It must not lower the threshold or choose between the tied values.

### Conclusion

Moving from the fixed 2-of-3 panel to the 3-of-5 quorum changes a correct
consensus into an abstention. The shared incorrect value of 0 does not win
either configuration.

These decisions were counted manually from the stored traces.
They still need to be checked using Team A's voting implementation.